# 👥 Notebook 3 — Perfil Estudiantil y Sistema de Alertas
**Metodología:** Ciclo praxeológico VER → JUZGAR → ACTUAR → DEVOLVER

---
## Objetivo
Construir un sistema de diagnóstico estudiantil basado en el momento **VER** del LEPI:
identificar perfiles, detectar estudiantes en riesgo y generar intervenciones personalizadas.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')

print("✅ Librerías cargadas — Portafolio Docente LEPI")
print("   Cristhian Harold Díaz Esteban | ORCID: 0009-0002-3524-6257")


In [ ]:
np.random.seed(42)
n = 80  # Replicando muestra de mi investigación (INNOVA, 2024)

df = pd.DataFrame({
    'estudiante_id':      [f'EST-{str(i).zfill(3)}' for i in range(1, n+1)],
    'carrera':            np.random.choice(['Adm. Empresas','Ing. Sistemas',
                                            'Educación','Psicología','Contaduría'], n),
    'semestre':           np.random.randint(1, 9, n),
    'asistencia_pct':     np.round(np.random.uniform(55, 100, n), 1),
    # Dimensiones LEPI del estudiante
    'lid_participacion':  np.round(np.random.normal(3.5, 0.8, n).clip(1,5), 1),
    'prax_reflexion':     np.round(np.random.normal(3.3, 0.9, n).clip(1,5), 1),
    'inn_creatividad':    np.round(np.random.normal(3.6, 0.7, n).clip(1,5), 1),
    # Rendimiento
    'nota_1':             np.round(np.random.normal(3.4, 0.8, n).clip(1,5), 1),
    'nota_2':             np.round(np.random.normal(3.6, 0.7, n).clip(1,5), 1),
    'pens_critico':       np.round(np.random.normal(3.5, 0.8, n).clip(1,5), 1),
    'inclusion':          np.round(np.random.normal(3.7, 0.7, n).clip(1,5), 1),
})

df['ILEPI'] = (
    df['lid_participacion']*0.35 +
    df['prax_reflexion']*0.35 +
    df['inn_creatividad']*0.30
).round(2)

df['promedio'] = df[['nota_1','nota_2']].mean(axis=1).round(2)

# Sistema de alertas (momento VER)
df['riesgo_score'] = (
    (df['asistencia_pct'] < 75).astype(int) +
    (df['promedio'] < 3.0).astype(int) +
    (df['ILEPI'] < 3.0).astype(int) +
    (df['pens_critico'] < 3.0).astype(int)
)
df['alerta'] = df['riesgo_score'].apply(
    lambda x: '🔴 Intervención urgente' if x >= 3 else
              ('🟡 Seguimiento' if x >= 1 else '🟢 En buen camino')
)

print(f"Grupo de {n} estudiantes (replica muestra investigación INNOVA 2024)")
print(f"\nDistribución de alertas:")
print(df['alerta'].value_counts())
print(f"\nEstudiantes que requieren intervención urgente: {(df['riesgo_score']>=3).sum()}")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    'Perfil Estudiantil LEPI — Momento VER\n'
    'Metodología: Diaz-Esteban (INNOVA 2024)',
    fontsize=13, fontweight='bold'
)

# Distribución ILEPI
axes[0,0].hist(df['ILEPI'], bins=20, color='#2E75B6', edgecolor='white', alpha=0.85)
axes[0,0].axvspan(1, 3.0, alpha=0.1, color='red')
axes[0,0].axvspan(3.0, 3.5, alpha=0.1, color='orange')
axes[0,0].axvspan(3.5, 5.0, alpha=0.1, color='green')
axes[0,0].axvline(df['ILEPI'].mean(), color='red', linestyle='--',
                   label=f"Media: {df['ILEPI'].mean():.2f}")
axes[0,0].set_title('Distribución ILEPI del Grupo'); axes[0,0].legend()

# Alertas
col_al = {'🟢 En buen camino':'#27AE60','🟡 Seguimiento':'#F39C12',
          '🔴 Intervención urgente':'#E74C3C'}
alertas = df['alerta'].value_counts()
axes[0,1].pie(alertas.values,
              labels=[l.split(' ',1)[1] for l in alertas.index],
              colors=[col_al[k] for k in alertas.index],
              autopct='%1.1f%%', startangle=90)
axes[0,1].set_title('Sistema de Alertas LEPI\n(Momento VER)')

# Pensamiento crítico vs ILEPI (dato de investigación: 87% mejora)
axes[0,2].scatter(df['ILEPI'], df['pens_critico'],
                  c=df['promedio'], cmap='RdYlGn', alpha=0.7, s=40)
r, p = __import__('scipy.stats', fromlist=['pearsonr']).pearsonr(df['ILEPI'], df['pens_critico'])
axes[0,2].set_xlabel('ILEPI'); axes[0,2].set_ylabel('Pensamiento Crítico')
axes[0,2].set_title(f'ILEPI ↔ Pensamiento Crítico\nr={r:.3f} {"✅" if p<0.05 else "⚠️"}')

# Por carrera
c_ilepi = df.groupby('carrera')['ILEPI'].mean().sort_values()
axes[1,0].barh(c_ilepi.index, c_ilepi.values,
               color=plt.cm.Blues(np.linspace(0.4,1.0,5)))
axes[1,0].axvline(3.5, color='red', linestyle='--', label='Meta 3.5')
axes[1,0].set_title('ILEPI por Carrera'); axes[1,0].legend()

# Heatmap dimensiones LEPI
dims = ['lid_participacion','prax_reflexion','inn_creatividad','pens_critico','inclusion']
corr = df[dims].corr()
sns.heatmap(corr, ax=axes[1,1], annot=True, fmt='.2f',
            cmap='RdYlGn', linewidths=0.5, vmin=0, vmax=1)
axes[1,1].set_title('Correlaciones LEPI')

# Asistencia vs Promedio por alerta
for alerta, grupo in df.groupby('alerta'):
    axes[1,2].scatter(grupo['asistencia_pct'], grupo['promedio'],
                      color=col_al.get(alerta,'gray'), alpha=0.7, s=40,
                      label=alerta.split(' ',1)[1][:15])
axes[1,2].axvline(75, color='red', linestyle=':', alpha=0.5)
axes[1,2].axhline(3.0, color='red', linestyle=':', alpha=0.5)
axes[1,2].set_xlabel('Asistencia %'); axes[1,2].set_ylabel('Promedio')
axes[1,2].set_title('Asistencia vs Rendimiento\n(semáforo de riesgo)')
axes[1,2].legend(fontsize=7)

plt.tight_layout()
plt.show()


In [ ]:
# ─── Rutas de intervención LEPI ───────────────────────
def ruta_intervencion_lepi(row):
    acciones = []
    if row['asistencia_pct'] < 75:
        acciones.append('Contactar estudiante (WhatsApp + email)')
    if row['prax_reflexion'] < 3.0:
        acciones.append('Asignar diario reflexivo semanal (momento JUZGAR)')
    if row['inn_creatividad'] < 3.0:
        acciones.append('Proyecto creativo individual + Design Thinking')
    if row['lid_participacion'] < 3.0:
        acciones.append('Rol de líder en equipo esta semana (momento ACTUAR)')
    if row['pens_critico'] < 3.0:
        acciones.append('Tutoría de pensamiento crítico + análisis de artículos')
    if not acciones:
        acciones.append('Designar como mentor de pares (replicar LEPI)')
    return ' | '.join(acciones)

df['ruta_lepi'] = df.apply(ruta_intervencion_lepi, axis=1)

# Top 5 estudiantes que más necesitan apoyo
top5 = df.nlargest(5, 'riesgo_score')[['estudiante_id','carrera','ILEPI','alerta','ruta_lepi']]
print("Top 5 estudiantes con plan de intervención LEPI:")
for _, row in top5.iterrows():
    print(f"\n{row['estudiante_id']} ({row['carrera']}) — ILEPI: {row['ILEPI']}")
    print(f"  {row['alerta']}")
    print(f"  Plan: {row['ruta_lepi']}")


## Conexión con mi práctica docente certificada

Este sistema de alertas refleja exactamente lo que implementé en:

**Pontificia Universidad Javeriana (2024)**  
Proyecto MEN-ICETEX: "Formación continua para educadores en servicio"  
→ Sistematización de asistencia + seguimiento a grupos de docentes + reportes periódicos

**UIS — Mentor Académico (2023–2024)**  
→ Rutas de aprendizaje personalizadas basadas en perfil del estudiante

**Corpoeducación — Programa La Candelita (2024)**  
→ Seguimiento telefónico + carga semanal de datos en plataforma = este mismo modelo
